# MicrosoftIQ - Companion Notebook

Companion to the deep-dive deck **Microsoft Foundry Microsoft IQ** (`decks/Microsoft Foundry Microsoft IQ.pptx`).
The notebook tests the Foundry IQ releases announced at Ignite in November 2026 — Foundry IQ is the new name of Azure AI Search on the new REST API versions and the latest Python SDK:

- **Search Management 2026-11-01** (stable control plane, including Serverless services)
- **Search Service 2026-10-01** (stable data plane) side by side with **2026-11-01-preview**
- Cursor pagination, find similar documents (`moreLikeThis`) and Content Understanding semantic chunking with image
  descriptions, all stable in 2026-10-01
- **Microsoft Retriever 1**, **knowledge base tools**, **knowledge graph retrieval** and the **Foundry IQ skills for GitHub Copilot CLI**, all preview in 2026-11-01-preview

One scenario runs through every step: an **HR policy assistant** for the fictional company Fabrikam,
with leave and travel policies for employees in Canada, Switzerland and Germany. The scenario checks if jurisdiction filters decide which policies an answer may use.

**Prerequisites**
- A Foundry IQ (Azure AI Search) service, **Basic tier or higher**, semantic ranker enabled, in a region that
  supports agentic retrieval; for step 8 (Microsoft Retriever 1) the service uses the **Standard** knowledge
  retrieval plan, as per official documentation
- Your own account has **Search Service Contributor**, **Search Index Data Contributor** and **Search Index Data Reader** on the search service, and **Reader** on its resource group for step 1; signed in with `az login`
- Optional, step 1: **Contributor** on a resource group to create a Serverless service (chargeable)
- Optional, step 6: a storage account with a container of a few PDF files that contain images, the search
  service's **system-assigned managed identity** with **Storage Blob Data Reader** on the account, and
  **Cognitive Services User** on the Foundry resource (extraction and image descriptions are billed there)
- Optional, steps 7 and 9: a Foundry project with a chat model deployment (`FOUNDRY_MODEL_NAME`) that knowledge
  bases support, and the search service identity with **Cognitive Services OpenAI User** on the Foundry resource
- Optional, step 10: a knowledge base with an entity or topic graph, built with the Foundry IQ skills for GitHub
  Copilot CLI (step 11) or the graph how-to pages

| Step | What it shows | Status | Deck slide |
|---|---|---|---|
| 0 | Settings, SDK version check and clients | GA | 5 |
| 1 | The search service on Search Management 2026-11-01; optional Serverless service | GA | 5, 9 |
| 2 | Sample index: twelve HR policies with a semantic configuration | GA | 6 |
| 3 | Stable and preview data-plane versions side by side | GA, preview | 9 |
| 4 | Cursor pagination for list operations | GA | 9 |
| 5 | Find similar documents (`moreLikeThis`) | GA | 9 |
| 6 | Content Understanding semantic chunking and image descriptions (optional) | GA | 9 |
| 7 | Knowledge base on the stable API: minimal effort, and low effort with a model (optional) | GA | 6, 9 |
| 8 | Microsoft Retriever 1: direct agentic search and query planning in a knowledge base | preview | 6, 9 |
| 9 | Knowledge base tools over REST and MCP; a model calling one tool (optional) | preview | 7, 9 |
| 10 | Knowledge graph retrieval: topic and entity graphs (optional query) | preview | 9 |
| 11 | Foundry IQ skills for GitHub Copilot CLI | preview | 9 |
| 12 | Cleanup | — | — |

Lines marked `# verify` use a shape that was documented at the time of writing but not executed here — check
them against the Learn page linked in the cell before you present.

## 0 · Settings, SDK version and clients  *(slide 5)*

One `.env` at the repository root, shared with the other kit notebooks. This notebook reuses
`AZURE_SEARCH_ENDPOINT` with its notebook 9 meaning and adds `NB11_*` settings, all optional except the
resource group used by step 1. Authentication is Microsoft Entra ID through `DefaultAzureCredential`; no keys.
Install the shared dependencies once from the repository root with `python3 -m pip install -r requirements.txt`
(see Local Setup in the kit README), then run with the repository root as the working directory.

The data-plane API versions are fixed in the notebook rather than in `.env`, because the comparison between them is
the point of several steps: **2026-10-01** is the stable contract, **2026-11-01-preview** adds the preview features.
`azure-search-documents` 12.2.0b1 is the first release whose default is 2026-11-01-preview; 12.1.0 targets
2026-10-01 and has no Retriever 1, knowledge base tool or graph classes. Each client below names its API version
explicitly, so the same package serves both columns.

Sources: https://learn.microsoft.com/en-us/azure/search/whats-new ·
https://pypi.org/project/azure-search-documents/#history ·
https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/search/azure-search-documents/CHANGELOG.md

In [1]:
import os
from pathlib import Path
from urllib.parse import urlparse
from dotenv import load_dotenv

ENV_FILE = Path(".env").resolve()
if not ENV_FILE.is_file():
    raise FileNotFoundError(f"No .env at {ENV_FILE}. Run from the repository root and save your .env there.")
load_dotenv(ENV_FILE, override=True)
print(f"Settings file: {ENV_FILE}")

def env(name, default=""):
    return (os.getenv(name) or default).strip()

AZURE_SEARCH_ENDPOINT = env("AZURE_SEARCH_ENDPOINT").rstrip("/")
AZURE_SUBSCRIPTION_ID = env("AZURE_SUBSCRIPTION_ID")
NB11_SEARCH_RESOURCE_GROUP = env("NB11_SEARCH_RESOURCE_GROUP")
NB11_SEARCH_MGMT_API_VERSION = env("NB11_SEARCH_MGMT_API_VERSION", "2026-11-01")
NB11_SERVERLESS_SERVICE_NAME = env("NB11_SERVERLESS_SERVICE_NAME")       # optional, step 1 (chargeable)
NB11_SERVERLESS_LOCATION = env("NB11_SERVERLESS_LOCATION")
NB11_CU_STORAGE_RESOURCE_ID = env("NB11_CU_STORAGE_RESOURCE_ID")         # optional, step 6
NB11_CU_CONTAINER = env("NB11_CU_CONTAINER")
NB11_GRAPH_KNOWLEDGE_BASE = env("NB11_GRAPH_KNOWLEDGE_BASE")             # optional, step 10
NB11_GRAPH_KNOWLEDGE_SOURCE = env("NB11_GRAPH_KNOWLEDGE_SOURCE")
# Optional, steps 6, 7 and 9c: a Foundry project and chat model deployment
FOUNDRY_PROJECT_ENDPOINT = env("FOUNDRY_PROJECT_ENDPOINT")
FOUNDRY_MODEL_NAME = env("FOUNDRY_MODEL_NAME")

GA_API = "2026-10-01"               # stable data plane
PREVIEW_API = "2026-11-01-preview"  # Retriever 1, knowledge base tools, knowledge graphs

required = {"AZURE_SEARCH_ENDPOINT": AZURE_SEARCH_ENDPOINT}
missing = [k for k, v in required.items() if not v or "<" in v]
LIVE = not missing                  # every step that calls Azure runs only when LIVE is True
MGMT_LIVE = LIVE and bool(AZURE_SUBSCRIPTION_ID and NB11_SEARCH_RESOURCE_GROUP and "<" not in AZURE_SUBSCRIPTION_ID)
MODEL_LIVE = LIVE and bool(FOUNDRY_PROJECT_ENDPOINT and FOUNDRY_MODEL_NAME and "<" not in FOUNDRY_PROJECT_ENDPOINT)
SEARCH_SERVICE = (urlparse(AZURE_SEARCH_ENDPOINT).hostname or "service").split(".")[0]
print("Required settings missing:", ", ".join(missing) if missing else "none")
print(f"Search service: {SEARCH_SERVICE} · management step: {'on' if MGMT_LIVE else 'off'} · model steps: {'on' if MODEL_LIVE else 'off'}")

Settings file: /workspaces/Microsoft-Foundry-What-s-New-and-Deep-Dive-Kit/.env
Required settings missing: none
Search service: gptkb-tb7tpjtuee4ji · management step: on · model steps: on


In [2]:
from importlib.metadata import version
from azure.search.documents.indexes import models as im

print("azure-search-documents", version("azure-search-documents"))
if not hasattr(im, "KnowledgeBaseMicrosoftRetriever1Model"):   # first shipped in 12.2.0b1
    print("This SDK has no preview classes for steps 3 and 8–10; install 12.2.0b1 or later with the install cell.")

try:
    from azure.search.documents import ApiVersion        # verify https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/search/azure-search-documents/CHANGELOG.md
    print("API versions known to this SDK:", ", ".join(v.value for v in ApiVersion)[-160:])
except ImportError:
    print("ApiVersion enum not exported by this SDK version")

azure-search-documents 12.2.0b1
API versions known to this SDK: 2020-06-30, 2023-11-01, 2024-07-01, 2025-09-01, 2026-04-01, 2026-08-01-preview, 2026-10-01, 2026-11-01-preview


### Clients

Two sets of data-plane clients, one per API version, with the same credential. `*_ga` clients send
`api-version=2026-10-01`; `*_pv` clients send `api-version=2026-11-01-preview`. The index name is fixed so that
every step works on the same twelve documents.

In [3]:
import json, time, re
import pandas as pd
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient
from azure.search.documents.indexes import models as im
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases import models as km

credential = DefaultAzureCredential()
INDEX = "nb11-hr-policies"
KS = "nb11-hr-ks"
KB_MINIMAL, KB_LOW, KB_RETRIEVER, KB_TOOLS = "nb11-hr-kb-minimal", "nb11-hr-kb-low", "nb11-hr-kb-retriever", "nb11-hr-kb-tools"

if LIVE:
    index_ga = SearchIndexClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=credential, api_version=GA_API)
    index_pv = SearchIndexClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=credential, api_version=PREVIEW_API)
    search_ga = SearchClient(endpoint=AZURE_SEARCH_ENDPOINT, index_name=INDEX, credential=credential, api_version=GA_API)
    search_pv = SearchClient(endpoint=AZURE_SEARCH_ENDPOINT, index_name=INDEX, credential=credential, api_version=PREVIEW_API)
    print("Clients ready for", AZURE_SEARCH_ENDPOINT)
else:
    print("Settings missing — clients not created")

def pick(d, name, default=None):
    # Responses come back as dicts with camelCase or snake_case keys depending on the SDK path.
    snake = re.sub(r"(?<!^)(?=[A-Z])", "_", name).lower()
    return d.get(name, d.get(snake, default)) if isinstance(d, dict) else default

def http_error(e):
    status = getattr(e, "status_code", None) or getattr(getattr(e, "response", None), "status_code", None)
    message = getattr(getattr(e, "error", None), "message", None) or str(e)
    return f"{status or type(e).__name__}: {message.splitlines()[0][:160]}"

Clients ready for https://gptkb-tb7tpjtuee4ji.search.windows.net


## 1 · The search service on Search Management 2026-11-01  *(slides 5, 9)*

Search Management 2026-11-01 is the new stable control-plane version, and as per official documentation it covers
the management of Foundry IQ Serverless services. Its service definition carries three properties that matter for
this notebook:

| Property | Values in 2026-11-01 |
|---|---|
| `sku.name` | Dedicated tiers such as `basic` or `standard`, and `serverless` (consumption-based, scales automatically) |
| `properties.computeType` | `Default` or `Confidential` (dedicated services on Azure confidential computing) |
| `properties.knowledgeRetrieval` | `free` or `standard`: the billing plan for agentic retrieval; step 8 needs `standard` |

No Python management package targets 2026-11-01 yet (`azure-mgmt-search` 10.0.0b2 is the newest, a preview
release), so this step calls Azure Resource Manager over REST with the API version in
`NB11_SEARCH_MGMT_API_VERSION`. The first cell only reads; the second creates a Serverless service when
`NB11_SERVERLESS_SERVICE_NAME` is set.

The status of Serverless differs between sources on 19 November 2026: the what's new page lists it as generally
available, while the Ignite blog post says general availability follows in December 2026. Check before presenting.

Sources: https://learn.microsoft.com/en-us/rest/api/searchmanagement/operation-groups?view=rest-searchmanagement-2026-11-01 ·
https://learn.microsoft.com/en-us/rest/api/searchmanagement/services/create-or-update?view=rest-searchmanagement-2026-11-01

In [4]:
import requests

ARM = "https://management.azure.com"
def arm_headers():
    return {"Authorization": "Bearer " + credential.get_token("https://management.azure.com/.default").token}

def service_url(resource_group, name):
    return (f"{ARM}/subscriptions/{AZURE_SUBSCRIPTION_ID}/resourceGroups/{resource_group}"
            f"/providers/Microsoft.Search/searchServices/{name}?api-version={NB11_SEARCH_MGMT_API_VERSION}")

def service_row(svc):
    p = svc.get("properties", {})
    return {"service": svc["name"], "sku": svc.get("sku", {}).get("name"), "location": svc.get("location"),
            "computeType": p.get("computeType"), "knowledgeRetrieval": p.get("knowledgeRetrieval"),
            "replicas": p.get("replicaCount"), "partitions": p.get("partitionCount"),
            "semanticSearch": p.get("semanticSearch"), "status": p.get("status")}

if MGMT_LIVE:
    r = requests.get(service_url(NB11_SEARCH_RESOURCE_GROUP, SEARCH_SERVICE), headers=arm_headers(), timeout=30)
    r.raise_for_status()
    this_service = r.json()
    display(pd.DataFrame([service_row(this_service)]))
    if this_service.get("properties", {}).get("knowledgeRetrieval") != "standard":
        print("The knowledge retrieval plan is not 'standard': step 8 (Microsoft Retriever 1) needs it, as per official documentation.")

    # List by resource group: the response pages with nextLink.
    url = (f"{ARM}/subscriptions/{AZURE_SUBSCRIPTION_ID}/resourceGroups/{NB11_SEARCH_RESOURCE_GROUP}"
           f"/providers/Microsoft.Search/searchServices?api-version={NB11_SEARCH_MGMT_API_VERSION}")
    rows = []
    while url:
        page = requests.get(url, headers=arm_headers(), timeout=30)
        page.raise_for_status()
        body = page.json()
        rows += [service_row(s) for s in body.get("value", [])]
        url = body.get("nextLink")
    display(pd.DataFrame(rows))
else:
    print("AZURE_SUBSCRIPTION_ID or NB11_SEARCH_RESOURCE_GROUP not set — step 1 skipped")

HTTPError: 400 Client Error: Bad Request for url: https://management.azure.com/subscriptions/dcbc681e-a69d-4f95-bc8e-da6054697474/resourceGroups/rg-rag-telemetry/providers/Microsoft.Search/searchServices/gptkb-tb7tpjtuee4ji?api-version=2026-11-01

In [5]:
print(r.status_code)
print(r.text)

400
{"error":{"code":"NoRegisteredProviderFound","message":"No registered resource provider found for location 'swedencentral' and API version '2026-11-01' for type 'searchServices'. The supported api-versions are '2014-07-31-Preview, 2015-02-28, 2015-08-19, 2019-10-01-Preview, 2020-03-13, 2020-08-01, 2020-08-01-Preview, 2021-04-01-Preview, 2021-06-06-Preview, 2022-09-01, 2023-11-01, 2024-03-01-Preview, 2024-06-01-Preview, 2025-02-01-Preview, 2025-05-01, 2026-03-01-Preview, 2026-09-01-Preview'. The supported locations are 'germanynorth, qatarcentral, polandcentral, italynorth, israelcentral, spaincentral, taiwannorth, newzealandnorth, jioindiacentral, malaysiawest, chilecentral, austriaeast, belgiumcentral, indiasouthcentral, denmarkeast, indonesiacentral, mexicocentral, southindia, swedencentral, ukwest, koreasouth, canadaeast, jioindiawest, westus3, germanywestcentral, norwayeast, switzerlandwest, switzerlandnorth, westus, westus2, eastus, eastus2, northeurope, westeurope, southeasta

### Optional: create a Serverless service

**Chargeable.** Creates a Serverless service named `NB11_SERVERLESS_SERVICE_NAME` in the same resource group, with
the request body of the documented *Create Or Update Service With Serverless* example, then waits for
provisioning. The rest of the notebook keeps using `AZURE_SEARCH_ENDPOINT`; point that setting at the new
service to repeat the data-plane steps on Serverless. An existing service with the same name is not changed.

In [ ]:
# CHARGEABLE: creates a Serverless search service (consumption-based billing)
if MGMT_LIVE and NB11_SERVERLESS_SERVICE_NAME:
    url = service_url(NB11_SEARCH_RESOURCE_GROUP, NB11_SERVERLESS_SERVICE_NAME)
    if requests.get(url, headers=arm_headers(), timeout=30).status_code == 200:
        print(NB11_SERVERLESS_SERVICE_NAME, "already exists — not changed")
    else:
        body = {"location": NB11_SERVERLESS_LOCATION or this_service["location"],
                "sku": {"name": "serverless"},
                "properties": {"hostingMode": "Default"}}
        r = requests.put(url, headers=arm_headers(), json=body, timeout=60)
        print("Create:", r.status_code)
        r.raise_for_status()
        for _ in range(60):
            svc = requests.get(url, headers=arm_headers(), timeout=30).json()
            state = svc.get("properties", {}).get("provisioningState")
            print("  provisioning state:", state, end="\r")
            if state and state.lower() in ("succeeded", "failed"):
                break
            time.sleep(20)
        print()
        display(pd.DataFrame([service_row(svc)]))
        print("Endpoint:", svc.get("properties", {}).get("endpoint"))
else:
    print("NB11_SERVERLESS_SERVICE_NAME not set — Serverless creation skipped")

## 2 · Sample index: twelve HR policies  *(slide 6)*

Twelve short policies for Fabrikam employees in Canada (`CA`), Switzerland (`CH`) and Germany (`DE`), plus
company-wide ones (`ALL`). One Canadian policy is marked `Superseded`, so later steps can show filters at work.
The index has text fields, filterable `jurisdiction`, `category` and `status` fields, and a default semantic
configuration, which knowledge bases and agentic retrieval use for ranking. It has no vector fields, so no
embedding deployment is needed.

The index is created on the stable API (2026-10-01). Source: https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-create-index

In [ ]:
POLICIES = [
    ("hr-001", "CA", "leave", "Current", "Parental leave — Canada",
     "Employees in Canada can take up to 18 months of parental leave. Fabrikam tops up government benefits to 90% of base salary for the first 16 weeks. Leave can start up to 4 weeks before the expected birth or adoption date."),
    ("hr-002", "CA", "leave", "Current", "How to request parental leave — Canada",
     "Request parental leave in the HR portal at least 8 weeks before the start date. Attach the expected date of birth or the adoption placement letter. Your manager and HR confirm within 5 business days."),
    ("hr-003", "CA", "vacation", "Current", "Vacation — Canada",
     "Employees in Canada receive 20 vacation days per year, rising to 25 days after 5 years of service. Up to 5 unused days can be carried over to the next year."),
    ("hr-004", "CH", "leave", "Current", "Maternity and paternity leave — Switzerland",
     "Employees in Switzerland receive 16 weeks of maternity leave at full salary and 4 weeks of paternity leave at full salary. Paternity leave can be taken within 12 months of the birth."),
    ("hr-005", "CH", "vacation", "Current", "Vacation — Switzerland",
     "Employees in Switzerland receive 25 vacation days per year, and 30 days from age 50. Vacation must be agreed with the manager at least 4 weeks in advance for periods longer than one week."),
    ("hr-006", "CH", "leave", "Current", "How to request leave — Switzerland",
     "Request maternity, paternity or unpaid leave in the HR portal and upload the medical certificate or birth certificate. HR Switzerland confirms within 3 business days."),
    ("hr-007", "DE", "leave", "Current", "Parental leave (Elternzeit) — Germany",
     "Employees in Germany can take up to 3 years of parental leave per child. Fabrikam pays a monthly supplement of 300 euros for the first 6 months of parental leave."),
    ("hr-008", "DE", "vacation", "Current", "Vacation — Germany",
     "Employees in Germany receive 30 vacation days per year. Unused days expire on 31 March of the following year unless illness prevented the vacation."),
    ("hr-009", "ALL", "work", "Current", "Remote work — all countries",
     "Employees can work remotely up to 3 days per week with their manager's agreement. Working from another country requires approval from HR and Tax at least 4 weeks in advance."),
    ("hr-010", "ALL", "travel", "Current", "Travel expenses — all countries",
     "Book business travel through the travel portal. Economy class applies to flights under 6 hours. Submit receipts within 30 days of returning."),
    ("hr-011", "CA", "leave", "Superseded", "Parental leave — Canada (2024 edition)",
     "Employees in Canada can take up to 12 months of parental leave. Fabrikam tops up government benefits to 80% of base salary for the first 12 weeks."),
    ("hr-012", "ALL", "leave", "Current", "HR portal — leave requests",
     "All leave requests go through the HR portal under My time off. Choose the leave type, the dates and attach supporting documents. Track the approval status on the same page."),
]
documents = [{"id": i, "jurisdiction": j, "category": c, "status": s, "title": t, "content": x}
             for i, j, c, s, t, x in POLICIES]

index_definition = im.SearchIndex(
    name=INDEX,
    fields=[
        im.SimpleField(name="id", type="Edm.String", key=True, filterable=True),
        im.SearchableField(name="title"),
        im.SearchableField(name="content"),
        im.SimpleField(name="jurisdiction", type="Edm.String", filterable=True, facetable=True),
        im.SimpleField(name="category", type="Edm.String", filterable=True, facetable=True),
        im.SimpleField(name="status", type="Edm.String", filterable=True, facetable=True),
    ],
    semantic_search=im.SemanticSearch(
        default_configuration_name="hr-semantic",
        configurations=[im.SemanticConfiguration(
            name="hr-semantic",
            prioritized_fields=im.SemanticPrioritizedFields(
                title_field=im.SemanticField(field_name="title"),
                content_fields=[im.SemanticField(field_name="content")]))]),
)
if LIVE:
    index_ga.create_or_update_index(index_definition)
    result = search_ga.merge_or_upload_documents(documents=documents)
    print(f"{INDEX}: {sum(r.succeeded for r in result)} of {len(documents)} documents uploaded")
    time.sleep(2)
    print("Document count:", search_ga.get_document_count())

## 3 · Stable and preview versions side by side  *(slide 9)*

As per official documentation, **2026-10-01** makes these capabilities generally available, among others:
model-backed query planning, answer synthesis and low and medium reasoning effort for knowledge bases; streaming
retrieve responses and cited-document lookup; file, indexed Azure SQL and indexed SharePoint knowledge sources;
cursor pagination; document-level permissions; Content Understanding semantic chunking and image descriptions;
and `moreLikeThis`. **2026-11-01-preview** adds Microsoft Retriever 1, knowledge base tools, knowledge graph
retrieval, OneLake permission preservation, GPT-6 knowledge base models and asynchronous file ingestion.

This step sends the same requests to both versions and records what each one accepts. It first creates the
knowledge source used from step 7 onwards (a stable object), then tries two preview-only definitions — a knowledge
base that plans with Microsoft Retriever 1, and one with a named tool — and deletes whatever the preview version
creates. A rejection on the stable version is the expected result, not a failure of the notebook.

Sources: https://learn.microsoft.com/en-us/azure/search/whats-new ·
https://learn.microsoft.com/en-us/rest/api/searchservice/operation-groups?view=rest-searchservice-2026-10-01 ·
https://learn.microsoft.com/en-us/rest/api/searchservice/operation-groups?view=rest-searchservice-2026-11-01-preview

In [ ]:
knowledge_source = im.SearchIndexKnowledgeSource(
    name=KS, description="Fabrikam HR policies for Canada, Switzerland, Germany and all countries.",
    search_index_parameters=im.SearchIndexKnowledgeSourceParameters(
        search_index_name=INDEX,
        source_data_fields=[im.SearchIndexFieldReference(name=f) for f in ("id", "title", "jurisdiction", "status")]))

def retriever_kb(name):
    return im.KnowledgeBase(name=name, knowledge_sources=[im.KnowledgeSourceReference(name=KS)],
                            models=[im.KnowledgeBaseMicrosoftRetriever1Model()],
                            retrieval_reasoning_effort=km.KnowledgeRetrievalMinimalReasoningEffort(),
                            output_mode=km.KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA)

def tool_kb(name):
    kb = im.KnowledgeBase(name=name, knowledge_sources=[im.KnowledgeSourceReference(name=KS)],
                          retrieval_reasoning_effort=km.KnowledgeRetrievalMinimalReasoningEffort(),
                          output_mode=km.KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA)
    kb.tools = [im.KnowledgeBaseTool(   # verify https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-configure-tools
        name="probe-tool", description="Probe tool.",
        knowledge_sources=[im.KnowledgeBaseToolKnowledgeSourceReference(name=KS)])]
    return kb

def attempt(label, version, action, cleanup=None):
    try:
        action()
        outcome = "accepted"
        if cleanup:
            cleanup()
    except Exception as e:
        outcome = "rejected — " + http_error(e)
    return {"request": label, "api-version": version, "outcome": outcome}

if LIVE:
    index_ga.create_or_update_knowledge_source(knowledge_source)
    rows = []
    for version, client, search in ((GA_API, index_ga, search_ga), (PREVIEW_API, index_pv, search_pv)):
        rows.append(attempt("Get service statistics", version, client.get_service_statistics))
        rows.append(attempt("Get knowledge source", version, lambda c=client: c.get_knowledge_source(KS)))
        rows.append(attempt("moreLikeThis query", version,
                            lambda s=search: list(s.search(search_text=None, more_like_this="hr-001", top=1))))
        rows.append(attempt("Knowledge base with Microsoft Retriever 1", version,
                            lambda c=client: c.create_or_update_knowledge_base(retriever_kb("nb11-probe-retriever")),
                            lambda c=client: c.delete_knowledge_base("nb11-probe-retriever")))
        rows.append(attempt("Knowledge base with a named tool", version,
                            lambda c=client: c.create_or_update_knowledge_base(tool_kb("nb11-probe-tools")),
                            lambda c=client: c.delete_knowledge_base("nb11-probe-tools")))
    display(pd.DataFrame(rows).pivot(index="request", columns="api-version", values="outcome"))

A rejection can come from the service (an HTTP 400 naming the unknown property) or from the SDK itself, which
can refuse a preview-only operation when the client is set to a stable version. Both show that the preview
feature is not part of the stable contract.

## 4 · Cursor pagination for list operations  *(slide 9)*

List operations on indexes, data sources, indexers, skillsets, synonym maps, aliases, knowledge sources and
knowledge bases now return pages. As per official documentation:

- `pageSize` sets the page size, 3,000 items by default and at most;
- `search` with `searchType=prefix` filters by a lowercase name prefix;
- every page but the last carries `@odata.nextLink`, which holds an opaque `$skiptoken`; follow it as it is,
  because combining `$skiptoken` with `search` or `pageSize` returns HTTP 400;
- responses carry no `@odata.count`.

The first cell uses the SDK (`list_indexes` with `page_size`, `search` and `search_type`, then `by_page`), the
second the same request over REST, to show the `@odata.nextLink` the SDK follows. A page size of 2 forces several
pages even on a small service.

Source: https://learn.microsoft.com/en-us/azure/search/search-how-to-page-list-results

In [ ]:
if LIVE:
    pages = index_ga.list_indexes(select=["name"], page_size=2).by_page()
    for n, page in enumerate(pages, 1):
        print(f"page {n}:", [ix.name for ix in page])
        if n == 1:
            token = pages.continuation_token
    print("continuation token after page 1:", (token or "")[:40], "…")

    # Resume from the token, then filter by prefix
    resumed = index_ga.list_indexes(select=["name"], page_size=2).by_page(continuation_token=token)
    print("resumed page:", [ix.name for ix in next(resumed)])
    print("prefix 'nb11-':", [ix.name for ix in index_ga.list_indexes(select=["name"], search="nb11-", search_type="prefix")])

In [ ]:
if LIVE:
    headers = {"Authorization": "Bearer " + credential.get_token("https://search.azure.com/.default").token}
    url = f"{AZURE_SEARCH_ENDPOINT}/indexes?api-version={GA_API}&pageSize=2&$select=name"
    n = 0
    while url and n < 10:
        n += 1
        body = requests.get(url, headers=headers, timeout=30).json()
        next_link = body.get("@odata.nextLink")
        print(f"page {n}: {[v['name'] for v in body.get('value', [])]} · nextLink: {'yes' if next_link else 'none (last page)'}")
        url = next_link

## 5 · Find similar documents  *(slide 9)*

`moreLikeThis` takes the key of a document and returns documents similar to it: the service derives query terms
from the source document's searchable fields (or the fields in `searchFields`) and leaves the source document
out of the results. It is generally available in 2026-10-01. As per official documentation, it can't be combined
with search text (not even `*`), with agentic queries or with Purview-enabled indexes, and an unknown key returns
an empty result.

The first query starts from the current Canadian parental leave policy and also finds the superseded one; the
second adds a filter, which applies to the similar documents as usual.

Source: https://learn.microsoft.com/en-us/azure/search/search-more-like-this

In [ ]:
def similar(key, filter=None):
    results = search_ga.search(search_text=None, more_like_this=key, search_fields=["content"],
                               select=["id", "title", "jurisdiction", "status"], filter=filter, top=4)
    return pd.DataFrame([{k: r[k] for k in ("id", "title", "jurisdiction", "status")} | {"score": round(r["@search.score"], 2)}
                         for r in results])

if LIVE:
    print("Similar to hr-001 (Parental leave — Canada):")
    display(similar("hr-001"))
    print("Same, current policies only:")
    display(similar("hr-001", "status eq 'Current'"))

## 6 · Content Understanding semantic chunking and image descriptions (optional)  *(slide 9)*

The Content Understanding skill extracts text, layout and images from files and returns chunks
(`text_sections`) with page numbers. Two options, introduced in 2026-05-01-preview, are generally available in
2026-10-01:

| Option | Setting in the skill |
|---|---|
| Semantic chunking | `chunkingProperties.method = "semantic"`, with `unit` (`characters` or `tokens`) and `maximumLength` |
| Image descriptions | `modelName` and `modelDeployment`, set together, with `extractionOptions` containing `images` |

The skill reads the file bytes (`file_data`), so the indexer sets `allowSkillsetToReadFileData`. Processing is
billed to the Foundry resource attached to the skillset; as per official documentation, the skill has no free daily
allowance. Notebook 9 uses the same skill with fixed-size chunks over OneLake; this step uses semantic chunks over
a blob container and writes one search document per chunk.

**Chargeable.** Runs only when `NB11_CU_STORAGE_RESOURCE_ID` and `NB11_CU_CONTAINER` are set; keep the container to
a few PDF files.

Sources: https://learn.microsoft.com/en-us/azure/search/cognitive-search-skill-content-understanding ·
https://learn.microsoft.com/en-us/azure/search/search-how-to-semantic-chunking-content-understanding

In [ ]:
CU_LIVE = LIVE and MODEL_LIVE and bool(NB11_CU_STORAGE_RESOURCE_ID and NB11_CU_CONTAINER)
CU_INDEX, CU_SOURCE, CU_SKILLSET, CU_INDEXER = "nb11-cu-chunks", "nb11-cu-blob", "nb11-cu-skillset", "nb11-cu-indexer"

if CU_LIVE:
    from azure.ai.projects import AIProjectClient
    project = AIProjectClient(endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=credential)
    CHAT_MODEL = project.deployments.get(FOUNDRY_MODEL_NAME).model_name   # model name, as in notebook 9
    ACCOUNT = (urlparse(FOUNDRY_PROJECT_ENDPOINT).hostname or "account").split(".")[0]
    FOUNDRY_SERVICES_URL = f"https://{ACCOUNT}.services.ai.azure.com"
    indexer_ga = SearchIndexerClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=credential, api_version=GA_API)

    context = "/document/text_sections/*"
    cu_index = im.SearchIndex(name=CU_INDEX, fields=[
        im.SearchableField(name="chunk_id", key=True, filterable=True, sortable=True, analyzer_name="keyword"),
        im.SimpleField(name="parent_id", type="Edm.String", filterable=True),
        im.SearchableField(name="chunk"),
        im.SearchableField(name="file_name", filterable=True),
        im.SimpleField(name="page_from", type="Edm.Int32", filterable=True, sortable=True),
        im.SimpleField(name="page_to", type="Edm.Int32", filterable=True, sortable=True)])
    datasource = im.SearchIndexerDataSourceConnection(
        name=CU_SOURCE, type="azureblob", connection_string=f"ResourceId={NB11_CU_STORAGE_RESOURCE_ID};",
        container=im.SearchIndexerDataContainer(name=NB11_CU_CONTAINER))
    cu_skill = im.ContentUnderstandingSkill(
        name="semantic-chunks", context="/document",
        extraction_options=["images", "locationMetadata"],
        model_name=CHAT_MODEL, model_deployment=FOUNDRY_MODEL_NAME,          # image descriptions
        chunking_properties=im.ContentUnderstandingSkillChunkingProperties(
            method="semantic", unit="tokens", maximum_length=500),          # verify https://learn.microsoft.com/en-us/azure/search/cognitive-search-skill-content-understanding
        inputs=[im.InputFieldMappingEntry(name="file_data", source="/document/file_data")],
        outputs=[im.OutputFieldMappingEntry(name="text_sections", target_name="text_sections"),
                 im.OutputFieldMappingEntry(name="normalized_images", target_name="normalized_images")])
    skillset = im.SearchIndexerSkillset(
        name=CU_SKILLSET, skills=[cu_skill],
        cognitive_services_account=im.AIServicesAccountIdentity(subdomain_url=FOUNDRY_SERVICES_URL),
        index_projection=im.SearchIndexerIndexProjection(
            selectors=[im.SearchIndexerIndexProjectionSelector(
                target_index_name=CU_INDEX, parent_key_field_name="parent_id", source_context=context,
                mappings=[im.InputFieldMappingEntry(name="chunk", source=f"{context}/content"),
                          im.InputFieldMappingEntry(name="page_from", source=f"{context}/locationMetadata/pageNumberFrom"),
                          im.InputFieldMappingEntry(name="page_to", source=f"{context}/locationMetadata/pageNumberTo"),
                          im.InputFieldMappingEntry(name="file_name", source="/document/metadata_storage_name")])],
            parameters=im.SearchIndexerIndexProjectionsParameters(projection_mode="skipIndexingParentDocuments")))
    indexer = im.SearchIndexer(
        name=CU_INDEXER, data_source_name=CU_SOURCE, target_index_name=CU_INDEX, skillset_name=CU_SKILLSET,
        parameters=im.IndexingParameters(configuration=im.IndexingParametersConfiguration(
            data_to_extract="contentAndMetadata", indexed_file_name_extensions=".pdf",
            allow_skillset_to_read_file_data=True)))
    print("Pipeline defined; run the next cell to create it.")
else:
    print("NB11_CU_STORAGE_RESOURCE_ID, NB11_CU_CONTAINER or the model settings not set — step 6 skipped")

In [ ]:
# CHARGEABLE: content extraction, semantic chunking and image descriptions on the Foundry resource
if CU_LIVE:
    index_ga.create_or_update_index(cu_index)
    indexer_ga.create_or_update_data_source_connection(datasource)
    indexer_ga.create_or_update_skillset(skillset)
    indexer_ga.create_or_update_indexer(indexer)       # a new indexer starts at once
    t0 = time.time()
    while time.time() - t0 < 1200:
        last = indexer_ga.get_indexer_status(CU_INDEXER).last_result
        state = last.status if last else "starting"
        print(f"  {int(time.time() - t0)} s · {state}", end="\r")
        if last and state != "inProgress":
            break
        time.sleep(15)
    print(f"\n{state}: {last.item_count if last else 0} files, {last.failed_item_count if last else 0} failed")
    chunks = SearchClient(endpoint=AZURE_SEARCH_ENDPOINT, index_name=CU_INDEX, credential=credential, api_version=GA_API)
    rows = [{"file": d["file_name"], "pages": f"{d['page_from']}–{d['page_to']}", "characters": len(d["chunk"] or ""),
             "start": (d["chunk"] or "")[:90].replace("\n", " ")}
            for d in chunks.search(search_text="*", select=["file_name", "page_from", "page_to", "chunk"], top=12)]
    display(pd.DataFrame(rows))

Semantic chunks follow the document's structure, so their lengths vary up to `maximumLength`; fixed-size chunks in
notebook 9 are all close to their limit. With a model set, the content of a chunk that holds a figure includes a
generated description of the image, so a text query can find a chart or diagram. Search for a word that only
appears in a figure to show it.

## 7 · Knowledge base on the stable API  *(slides 6, 9)*

A **knowledge source** describes where content lives (here, the step 2 index); a **knowledge base** plans the
queries, runs them against its sources and returns grounding data or an answer with citations. On 2026-10-01:

- **Minimal** reasoning effort needs no model: the request carries one or more semantic *intents*, and the
  knowledge base returns extracted content (`extractiveData`) with references.
- **Low** and **medium** effort use a model to plan sub-queries from the conversation (*messages*) and can
  synthesise an answer (`answerSynthesis`). This is now generally available.

The first cell creates the minimal knowledge base with the SDK model classes; the second, when a model is
configured, creates a low-effort knowledge base from its REST definition, as notebook 9 does.

Sources: https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-create-knowledge-base ·
https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-retrieve

In [ ]:
QUESTIONS = [
    "What parental leave do Canadian employees receive and how do they request it?",
    "How much paternity leave is there in Switzerland?",
    "Can I carry over vacation days in Germany?",
]

def show(result, label=""):
    result = result.as_dict() if hasattr(result, "as_dict") else result
    text = " ".join(pick(c, "text", "") for m in pick(result, "response", []) for c in pick(m, "content", []))
    refs = [pick(r, "docKey") or pick(pick(r, "sourceData") or {}, "id") for r in pick(result, "references", [])]
    activity = [(pick(a, "type"), pick(pick(a, "model") or {}, "modelName")) for a in pick(result, "activity", [])]
    print(f"{label} references: {refs}")
    print("   activity:", [t if not m else f"{t} ({m})" for t, m in activity])
    print("   content:", text[:300].replace("\n", " "), "…\n")
    return {"references": len(refs), "documents": ", ".join(sorted({r for r in refs if r}))}

if LIVE:
    index_ga.create_or_update_knowledge_base(im.KnowledgeBase(
        name=KB_MINIMAL, knowledge_sources=[im.KnowledgeSourceReference(name=KS)],
        retrieval_reasoning_effort=km.KnowledgeRetrievalMinimalReasoningEffort(),
        output_mode=km.KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA))
    kb_minimal = KnowledgeBaseRetrievalClient(endpoint=AZURE_SEARCH_ENDPOINT, knowledge_base_name=KB_MINIMAL,
                                              credential=credential, api_version=GA_API)
    for q in QUESTIONS[:1]:
        show(kb_minimal.retrieve(km.KnowledgeBaseRetrievalRequest(
            intents=[km.KnowledgeRetrievalSemanticIntent(search=q)], include_activity=True)), "minimal")

In [ ]:
# CHARGEABLE: query planning and answer synthesis on the chat model
if LIVE and MODEL_LIVE:
    from azure.ai.projects import AIProjectClient
    project = AIProjectClient(endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=credential)
    CHAT_MODEL = project.deployments.get(FOUNDRY_MODEL_NAME).model_name
    ACCOUNT = (urlparse(FOUNDRY_PROJECT_ENDPOINT).hostname or "account").split(".")[0]
    model = {"kind": "azureOpenAI", "azureOpenAIParameters": {
        "resourceUri": f"https://{ACCOUNT}.openai.azure.com", "deploymentId": FOUNDRY_MODEL_NAME, "modelName": CHAT_MODEL}}
    index_ga.create_or_update_knowledge_base({
        "name": KB_LOW, "knowledgeSources": [{"name": KS}], "models": [model],
        "retrievalReasoningEffort": {"kind": "low"}, "outputMode": "answerSynthesis",
        "answerInstructions": "Answer in at most three sentences, cite every policy, and ignore superseded policies."})
    kb_low = KnowledgeBaseRetrievalClient(endpoint=AZURE_SEARCH_ENDPOINT, knowledge_base_name=KB_LOW,
                                          credential=credential, api_version=GA_API)
    show(kb_low.retrieve({"messages": [{"role": "user", "content": [{"type": "text", "text": QUESTIONS[0]}]}],
                          "includeActivity": True}), "low")
else:
    print("FOUNDRY_PROJECT_ENDPOINT or FOUNDRY_MODEL_NAME not set — low-effort knowledge base skipped")

## 8 · Microsoft Retriever 1 (preview)  *(slides 6, 9)*

Microsoft Retriever 1 is a retrieval model hosted by Foundry IQ. It plans queries and searches iteratively,
without an external query-planning model or a model connection. It works in two places:

| Where | How it is switched on |
|---|---|
| Direct index search | `queryType: "agentic"` with `agenticOptions` on a search request |
| Knowledge base | `models: [{"kind": "microsoftRetriever1"}]` with minimal reasoning effort |

As per official documentation, it needs the **Standard** knowledge retrieval plan (step 1), works through the SDK
and REST but not in the portals, and has these limits: a knowledge base that uses only Retriever 1 doesn't support
low or medium effort or knowledge source selection; several intents in one request skip Retriever 1 and return
HTTP 206; direct agentic search rejects a nonzero `skip`, `orderby`, `count`, facets and semantic captions or
answers. *Auto* effort combines Retriever 1 with an external model and is out of scope here.

**Chargeable.** Retrieval reasoning tokens are billed on the Standard plan.

Source: https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-use-microsoft-retriever

### 8a · Direct agentic search

The response is a normal result list. The debug information shows why the planning stopped (`modelStopped`,
`maxRuntimeExceeded` or `modelFailed`), the sub-queries and the planning and reasoning tokens. HTTP 206 means partial
success: documents may still be returned.

In [ ]:
# CHARGEABLE: Microsoft Retriever 1 planning and reasoning tokens
from azure.search.documents.models import AgenticSearchOptions, QueryDebugMode, QueryType

if LIVE:
    results = search_pv.search(
        search_text=QUESTIONS[0], query_type=QueryType.AGENTIC,
        agentic_options=AgenticSearchOptions(max_runtime_in_seconds=60),
        filter="status eq 'Current'", select=["id", "title", "jurisdiction"], top=5,
        debug=QueryDebugMode.AGENTIC)
    display(pd.DataFrame([{k: r[k] for k in ("id", "title", "jurisdiction")} for r in results]))
    print(json.dumps(results.get_debug_info(), indent=1, default=str)[:1200])   # verify https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-use-microsoft-retriever

### 8b · Retriever 1 in a knowledge base, compared with minimal retrieval

The knowledge base from step 3's probe, now kept: same knowledge source and output mode as step 7's minimal
knowledge base, plus the Retriever 1 model. The activity of each request shows a `modelQueryPlanning` entry with
model `microsoft-retriever-1` when Retriever 1 planned the queries. The table compares both knowledge bases on the
three questions: elapsed time, number of references and the documents cited.

In [ ]:
# CHARGEABLE: Microsoft Retriever 1 planning tokens on three questions
if LIVE:
    index_pv.create_or_update_knowledge_base(retriever_kb(KB_RETRIEVER))
    kb_retriever = KnowledgeBaseRetrievalClient(endpoint=AZURE_SEARCH_ENDPOINT, knowledge_base_name=KB_RETRIEVER,
                                                credential=credential, api_version=PREVIEW_API)
    kb_minimal_pv = KnowledgeBaseRetrievalClient(endpoint=AZURE_SEARCH_ENDPOINT, knowledge_base_name=KB_MINIMAL,
                                                 credential=credential, api_version=PREVIEW_API)
    rows = []
    for q in QUESTIONS:
        for label, client in (("minimal", kb_minimal_pv), ("Retriever 1", kb_retriever)):
            request = km.KnowledgeBaseRetrievalRequest(
                intents=[km.KnowledgeRetrievalSemanticIntent(search=q)],
                retrieval_reasoning_effort=km.KnowledgeRetrievalMinimalReasoningEffort(),
                output_mode=km.KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA, include_activity=True)
            t0 = time.perf_counter()
            result = client.retrieve(request)
            seconds = round(time.perf_counter() - t0, 1)
            rows.append({"question": q[:45], "knowledge base": label, "seconds": seconds, **show(result, label)})
    display(pd.DataFrame(rows))

## 9 · Knowledge base tools over REST and MCP (preview)  *(slides 7, 9)*

A **knowledge base tool** is a named retrieval entry point in a knowledge base: its own description, its own subset
of knowledge sources with filters, and default request settings. An agent sees each tool by name, so one knowledge
base can offer *canada-hr-search* and *switzerland-hr-search* without two copies of the index. As per official
documentation:

- a request names the tool with `toolName`; an empty or unknown name fails instead of falling back to the whole
  knowledge base;
- request values override the tool's defaults, but a request filter is combined with the tool's filter (AND) and
  can't replace it or add knowledge sources;
- the knowledge base MCP endpoint at 2026-11-01-preview lists each tool under its configured name; earlier API
  versions only offer `knowledge_base_retrieve`.

Source: https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-configure-tools

### 9a · Define three tools and call them over REST

In [ ]:
def hr_tool(name, description, jurisdiction):
    return im.KnowledgeBaseTool(   # verify https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-configure-tools
        name=name, description=description,
        knowledge_sources=[im.KnowledgeBaseToolKnowledgeSourceReference(
            name=KS, filter_add_on=f"(jurisdiction eq '{jurisdiction}' or jurisdiction eq 'ALL') and status eq 'Current'",
            reranker_threshold=0, max_output_documents=50, fail_on_error=True)],
        retrieval_reasoning_effort=km.KnowledgeRetrievalMinimalReasoningEffort(),
        retrieve_defaults=im.KnowledgeBaseRetrieveDefaults(
            max_runtime_in_seconds=30, max_output_size_in_tokens=5000, max_output_documents=12))

TOOLS = {"canada-hr-search": ("Find current HR policies for employees in Canada.", "CA"),
         "switzerland-hr-search": ("Find current HR policies for employees in Switzerland.", "CH"),
         "germany-hr-search": ("Find current HR policies for employees in Germany.", "DE")}

if LIVE:
    kb = im.KnowledgeBase(name=KB_TOOLS, knowledge_sources=[im.KnowledgeSourceReference(name=KS)],
                          retrieval_reasoning_effort=km.KnowledgeRetrievalMinimalReasoningEffort(),
                          output_mode=km.KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA)
    kb.tools = [hr_tool(n, d, j) for n, (d, j) in TOOLS.items()]
    index_pv.create_or_update_knowledge_base(kb)
    kb_tools = KnowledgeBaseRetrievalClient(endpoint=AZURE_SEARCH_ENDPOINT, knowledge_base_name=KB_TOOLS,
                                            credential=credential, api_version=PREVIEW_API)

    def via_tool(tool, question, filter_add_on=None):
        params = ([km.SearchIndexKnowledgeSourceParams(knowledge_source_name=KS, filter_add_on=filter_add_on)]
                  if filter_add_on else None)
        return kb_tools.retrieve(km.KnowledgeBaseRetrievalRequest(
            tool_name=tool, intents=[km.KnowledgeRetrievalSemanticIntent(search=question)],
            knowledge_source_params=params, include_activity=True))

    q = "How long is parental leave?"
    for tool in TOOLS:
        show(via_tool(tool, q), tool)
    print("Request filter combined with the tool filter (AND):")
    show(via_tool("canada-hr-search", "leave and vacation", "category eq 'vacation'"), "canada-hr-search + vacation")
    print("Unknown tool name:")
    try:
        via_tool("france-hr-search", q)
    except Exception as e:
        print("  rejected —", http_error(e))

### 9b · Discover and call the tools over MCP

The same knowledge base as an MCP server: `tools/list` returns the three tools with their descriptions and input
schemas, and `tools/call` runs one. Authentication is a Microsoft Entra token for `https://search.azure.com`; the
caller needs **Search Index Data Reader**. The call result holds the extracted content as text, without the
activity and references arrays of the REST response. The client code follows notebooks 1, 8 and 9
(`streamablehttp_client` from `mcp` 1.x).

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

KB_MCP_URL = f"{AZURE_SEARCH_ENDPOINT}/knowledgebases/{KB_TOOLS}/mcp?api-version={PREVIEW_API}"

if LIVE:
    headers = {"Authorization": "Bearer " + credential.get_token("https://search.azure.com/.default").token}
    async with streamablehttp_client(KB_MCP_URL, headers=headers) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            mcp_tools = (await session.list_tools()).tools
            display(pd.DataFrame([t.model_dump(include={"name", "description", "inputSchema"}) for t in mcp_tools]))

            tool = next(t for t in mcp_tools if t.name == "switzerland-hr-search")
            schema = tool.inputSchema or {}
            # Fill the tool's first required string argument with the question; the schema names it.
            arg = next((k for k in schema.get("required", []) if schema["properties"][k].get("type") == "string"),
                       next(iter(schema.get("properties", {})), "query"))
            result = await session.call_tool(tool.name, {arg: "How much paternity leave is there?"})   # verify https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-configure-tools
            print(f"{tool.name}({arg}=…):", " ".join(getattr(c, "text", "") for c in result.content)[:600])

### 9c · A model calling one tool (optional)

An agent or model uses the tool through the Responses API `mcp` tool type, restricted with `allowed_tools` to the
tool for the employee's country. The search token is passed in the tool's headers, as in the documented example;
it expires after about an hour, so production agents use a project connection instead (notebook 9, step 5).

**Chargeable.** One model call.

In [ ]:
# CHARGEABLE: one model call
if LIVE and MODEL_LIVE:
    from azure.ai.projects import AIProjectClient
    project = AIProjectClient(endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=credential)
    openai_client = project.get_openai_client()
    token = credential.get_token("https://search.azure.com/.default").token
    response = openai_client.responses.create(
        model=FOUNDRY_MODEL_NAME,
        input="I work for Fabrikam in Zurich. How much paternity leave do I get and how do I request it?",
        tools=[{"type": "mcp", "server_label": "fabrikam_hr", "server_url": KB_MCP_URL,
                "allowed_tools": ["switzerland-hr-search"], "headers": {"Authorization": f"Bearer {token}"},
                "require_approval": "never"}])   # verify https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-configure-tools
    print([item.type for item in response.output])
    print(response.output_text)
else:
    print("FOUNDRY_PROJECT_ENDPOINT or FOUNDRY_MODEL_NAME not set — step 9c skipped")

## 10 · Knowledge graph retrieval (preview)  *(slide 9)*

Two graph features, both in 2026-11-01-preview:

| Graph | What it adds | Built from |
|---|---|---|
| Topic graph | Broader evidence across themes for diverse questions | The existing embeddings of a search index knowledge source, plus a topic index |
| Entity graph | Answers that follow connections between specific entities | A blob knowledge source with graph extraction and a JSON-LD ontology |

As per official documentation:

- A **topic graph** is set on a search index knowledge source with `topicGraph` (`balanced` by default) and can be
  overridden per request (`diverse` with a `diversityFactor`, or `none`). It needs a populated topic index, which
  the Foundry IQ skills for GitHub Copilot CLI prepare (step 11). The knowledge base doesn't detect diverse
  questions on its own: the request sets `diverse`.
- An **entity graph** is extracted at ingestion from an Azure Blob Storage container (not Azure Data Lake Storage
  Gen2, with soft delete on and versioning off) using a chat model, an embedding model and an ontology. It needs a
  **new** knowledge base with an Azure OpenAI model and **low** effort; it can't be enabled on an existing one. It
  is API-only in this release. There's no charge for the graph feature itself during preview; the model tokens
  are billed.
- Graphs can also be queried directly with GQL through `docs/search.graph` (`SearchClient.graph` in 12.2.0b1).

The twelve short policies of this notebook are too small for a useful graph, so the code below queries a graph
knowledge base built separately, when `NB11_GRAPH_KNOWLEDGE_BASE` and `NB11_GRAPH_KNOWLEDGE_SOURCE` are set. It asks
for the graph query that was run and hydrated evidence documents. The `select` fields are those of the documented
entity graph example; change them to the fields of your graph index.

Sources: https://learn.microsoft.com/en-us/azure/search/knowledge-graph-features-overview ·
https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-entity-graph ·
https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-topic-graph

In [ ]:
# CHARGEABLE: query planning on the graph knowledge base's model
GRAPH_QUESTION = "Which teams depend on people who report to the same manager?"   # adapt to your graph

if LIVE and NB11_GRAPH_KNOWLEDGE_BASE and NB11_GRAPH_KNOWLEDGE_SOURCE:
    kb_graph = KnowledgeBaseRetrievalClient(endpoint=AZURE_SEARCH_ENDPOINT, knowledge_base_name=NB11_GRAPH_KNOWLEDGE_BASE,
                                            credential=credential, api_version=PREVIEW_API)
    request = {   # verify https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-entity-graph
        "messages": [{"role": "user", "content": [{"type": "text", "text": GRAPH_QUESTION}]}],
        "retrievalReasoningEffort": {"kind": "low"},
        "includeActivity": True,
        "knowledgeSourceParams": [{
            "knowledgeSourceName": NB11_GRAPH_KNOWLEDGE_SOURCE, "kind": "azureBlob",
            "includeReferences": True, "includeReferenceSourceData": True,
            "graphParameters": {"includeQuery": True,
                                "provenance": {"kind": "hydrate",
                                               "hydrateDocuments": {"select": "uid,snippet,snippet_parent_id,blob_url", "top": 10}}}}],
    }
    result = kb_graph.retrieve(request).as_dict()
    show(result, "graph")
    for a in pick(result, "activity", []):
        if "graph" in json.dumps(a).lower():
            print(json.dumps(a, indent=1, default=str)[:1200])
else:
    print("NB11_GRAPH_KNOWLEDGE_BASE or NB11_GRAPH_KNOWLEDGE_SOURCE not set — step 10 query skipped")

## 11 · Foundry IQ skills for GitHub Copilot CLI (preview)  *(slide 9)*

*Explain-only: these commands run in a terminal, not in the notebook.*

The `foundry-iq-skills` plugin gives GitHub Copilot CLI one skill, `foundry-iq`, that builds and checks Foundry
IQ objects from a conversation: knowledge sources over local files, Azure Blob Storage or Azure Data Lake Storage,
knowledge bases with retrieval validation, connections to existing agents, diagnosis and cleanup planning, and
entity graphs, topic graphs and ontologies. As per official documentation, it asks for approval before it creates
a service, knowledge source or knowledge base, connects a knowledge base to an agent, or makes another planned
resource change.

Prerequisites: Git, GitHub Copilot CLI on Windows or Linux, Python 3.12 or later, Azure CLI signed in.

Inside Copilot CLI:

```text
/plugin marketplace add microsoft/azure-skills
/plugin install foundry-iq-skills@azure-skills
/plugin update foundry-iq-skills@azure-skills
```

Or from a shell:

```bash
copilot plugin marketplace add microsoft/azure-skills
copilot plugin install foundry-iq-skills@azure-skills
copilot plugin list
```

Prompts that continue this notebook, run in the repository folder:

- *"Query the knowledge base nb11-hr-kb-tools on my search service with citations."*
- *"Show me what Foundry IQ graph can do."*
- *"Build a topic graph for the index nb11-hr-policies and tell me what it needs."*

Sources: https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-install-foundry-iq-skills ·
https://learn.microsoft.com/en-us/azure/developer/azure-skills/plugins/foundry-iq/foundry-iq

## 12 · Cleanup

Deletes what this notebook created on the search service: the four knowledge bases, the knowledge source, the
step 2 index, and the step 6 indexer, skillset, data source and index. Set `CLEANUP = True` to run it.

**Not deleted:** the search service itself (billed per hour on Basic and higher tiers), the Serverless service
from step 1 if you created one (delete it in the Azure portal; it bills by consumption), the knowledge retrieval
plan setting, the storage account and its files from step 6, the graph knowledge base of step 10 and anything the
Copilot CLI plugin created in step 11, and the Foundry model deployments.

In [ ]:
# DESTRUCTIVE: deletes the search objects created in steps 2–9
CLEANUP = False
if LIVE and CLEANUP:
    indexer_cleanup = SearchIndexerClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=credential, api_version=GA_API)
    for action, name in [(index_pv.delete_knowledge_base, KB_TOOLS), (index_pv.delete_knowledge_base, KB_RETRIEVER),
                         (index_ga.delete_knowledge_base, KB_LOW), (index_ga.delete_knowledge_base, KB_MINIMAL),
                         (index_ga.delete_knowledge_source, KS),
                         (indexer_cleanup.delete_indexer, CU_INDEXER), (indexer_cleanup.delete_skillset, CU_SKILLSET),
                         (indexer_cleanup.delete_data_source_connection, CU_SOURCE), (index_ga.delete_index, CU_INDEX),
                         (index_ga.delete_index, INDEX)]:
        try:
            action(name)
            print("deleted", name)
        except Exception as e:
            print("skipped", name, "-", type(e).__name__)

## What you built

| Step | Built / shown | Azure resource touched |
|---|---|---|
| 0 | Settings, SDK version and two sets of clients (2026-10-01 and 2026-11-01-preview) | None |
| 1 | Service properties on Search Management 2026-11-01; optional Serverless service | Search service (read); resource group (optional create) |
| 2 | Index `nb11-hr-policies` with twelve policies | Search service |
| 3 | Knowledge source `nb11-hr-ks`; a table of what each API version accepts | Search service |
| 4 | Pages of the index list through the SDK and REST | Search service (read) |
| 5 | Similar documents for one policy, with and without a filter | Search service (read) |
| 6 | Semantic chunks with page numbers and image descriptions in `nb11-cu-chunks` (optional) | Search service; storage account (read); Foundry resource |
| 7 | Knowledge bases `nb11-hr-kb-minimal` and `nb11-hr-kb-low` (optional) | Search service; Foundry resource (chat model) |
| 8 | Direct agentic search and knowledge base `nb11-hr-kb-retriever` with Microsoft Retriever 1 | Search service (Standard knowledge retrieval plan) |
| 9 | Knowledge base `nb11-hr-kb-tools` with three tools, called over REST, MCP and by a model | Search service; Foundry project (optional model call) |
| 10 | One retrieve request on a graph knowledge base (optional) | Search service; Foundry resource |
| 11 | Commands and prompts for the Copilot CLI plugin | None from the notebook |

## Sources and tested versions

Written against `azure-search-documents` 12.2.0b1 (default REST API version 2026-11-01-preview; every client here
names its version), Search Management REST API 2026-11-01 over `requests`, `mcp` 1.x (`streamablehttp_client`, as in
notebooks 1, 8 and 9), `azure-ai-projects` 2.10.0 and `openai` 3.x (steps 6, 7 and 9c). **Not executed against Azure.**

- https://learn.microsoft.com/en-us/azure/search/whats-new
- https://learn.microsoft.com/en-us/rest/api/searchmanagement/operation-groups?view=rest-searchmanagement-2026-11-01
- https://learn.microsoft.com/en-us/rest/api/searchmanagement/services/create-or-update?view=rest-searchmanagement-2026-11-01
- https://learn.microsoft.com/en-us/rest/api/searchservice/operation-groups?view=rest-searchservice-2026-10-01
- https://learn.microsoft.com/en-us/rest/api/searchservice/operation-groups?view=rest-searchservice-2026-11-01-preview
- https://learn.microsoft.com/en-us/azure/search/search-how-to-page-list-results
- https://learn.microsoft.com/en-us/azure/search/search-more-like-this
- https://learn.microsoft.com/en-us/azure/search/cognitive-search-skill-content-understanding
- https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-create-knowledge-base
- https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-retrieve
- https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-use-microsoft-retriever
- https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-configure-tools
- https://learn.microsoft.com/en-us/azure/search/knowledge-graph-features-overview
- https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-entity-graph
- https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-topic-graph
- https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-install-foundry-iq-skills
- https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/search/azure-search-documents/CHANGELOG.md
- https://github.com/microsoft/iq-series

## Before you build on this: re-check …

- **`azure-search-documents` 12.2.0b1**: PyPI lists it on 16 November 2026 while the changelog on the main branch
  still says *Unreleased*; later betas may rename preview classes (`KnowledgeBaseTool`,
  `KnowledgeBaseMicrosoftRetriever1Model`, the graph parameter classes). The kit caps it below 12.3.
- **Serverless status** (step 1): generally available on the what's new page, December 2026 in the Ignite blog.
- **`# verify` lines**: the semantic chunking keyword names (step 6), the debug information shape (step 8a), the
  tool classes and the MCP argument name (step 9), the `mcp` tool type through the project's OpenAI client (9c)
  and the graph request (step 10).
- **Retriever 1 regions and price**: the documentation names a "supported region" without a list.
- **Preview limits** of Retriever 1, tools and graphs, expected to change before general availability.